# F1 Lakehouse — Exploration Notebook

A scratchpad for poking at the pipeline while you build it.

**What's in here:**
1. Setup — imports + project paths
2. Inspect the `raw_races` Delta table that Dagster wrote
3. Explore the source API directly (Jolpica)
4. Query Delta from DuckDB (same engine dbt uses)
5. Delta time travel — read past versions

**Run it:** `uv run jupyter lab` from project root.

## Setup

In [ ]:
from pathlib import Path

import duckdb
import httpx
import polars as pl
from deltalake import DeltaTable

# Resolve project root whether notebook is launched from root or from notebooks/
PROJECT_ROOT = Path.cwd() if (Path.cwd() / "pyproject.toml").exists() else Path.cwd().parent
DATA_ROOT = PROJECT_ROOT / "data"

# The lakehouse is namespaced by dataset (the stack/dataset split — see CLAUDE.md).
# Every zone lives under data/<zone>/<dataset>/, so paths are built from DATASET.
# Point this at another dataset (e.g. "nyc_taxi") to explore it with the same cells.
DATASET = "f1"

# Bronze (Delta — PolarsDeltaIOManager appends '.delta' as a format marker)
RAW_RACES_PATH   = DATA_ROOT / "raw" / DATASET / "raw_races.delta"
RAW_DRIVERS_PATH = DATA_ROOT / "raw" / DATASET / "raw_drivers.delta"
RAW_RESULTS_PATH = DATA_ROOT / "raw" / DATASET / "raw_results.delta"

# Silver (Parquet — dbt external materialization)
STG_RACES_PATH   = DATA_ROOT / "staging" / DATASET / "stg_races.parquet"
STG_DRIVERS_PATH = DATA_ROOT / "staging" / DATASET / "stg_drivers.parquet"
STG_RESULTS_PATH = DATA_ROOT / "staging" / DATASET / "stg_results.parquet"

# Gold (Parquet — dbt external materialization)
MART_COUNTRIES_PATH = DATA_ROOT / "marts" / DATASET / "mart_country_race_summary.parquet"
MART_STANDINGS_PATH = DATA_ROOT / "marts" / DATASET / "mart_driver_standings.parquet"

print(f"Project root: {PROJECT_ROOT}")
print(f"Data root:    {DATA_ROOT}")
print(f"Dataset:      {DATASET}")
print(f"Polars:       {pl.__version__}")
print(f"DuckDB:       {duckdb.__version__}")

## 1. Inspect the raw Delta table

Read `data/raw/f1/raw_races.delta/` — the Delta table Polars wrote during the last Dagster materialization (namespaced under the `f1/` dataset).

In [ ]:
# Read the whole Delta table into a Polars DataFrame
df = pl.read_delta(str(RAW_RACES_PATH))
df.head()

In [ ]:
# Schema (column names + Polars dtypes). Nested objects from the API show up as Struct columns.
df.schema

In [ ]:
# Row count + null counts per column — useful before designing staging models
print(f"Rows: {df.height}")
df.null_count()

## 2. Explore the source API

Sometimes the fastest way to understand a source is to just `GET` it and stare at one record.

In [ ]:
# Pull one season of races straight from Jolpica
response = httpx.get("https://api.jolpi.ca/ergast/f1/2024/races.json", timeout=30.0)
response.raise_for_status()

payload = response.json()["MRData"]
print(f"Total races reported: {payload['total']}")
print(f"Top-level keys:       {list(payload['RaceTable']['Races'][0].keys())}")

In [ ]:
# Pretty-print one raw race record to understand the JSON shape
import json
print(json.dumps(payload["RaceTable"]["Races"][0], indent=2))

## 3. Query the Delta table from DuckDB

DuckDB reads Delta tables natively via its `delta` extension. This is what dbt-duckdb uses under the hood — useful for prototyping a staging query before committing it to a dbt model.

In [ ]:
conn = duckdb.connect()
conn.execute("INSTALL delta; LOAD delta;")

# Identifier-quoting note: Ergast JSON uses camelCase keys (raceName, etc.),
# so we keep them quoted in SQL. dbt staging models will rename to snake_case.
conn.execute(f"""
    SELECT season, round, "raceName" AS race_name, "date" AS race_date
    FROM delta_scan('{RAW_RACES_PATH}')
    ORDER BY CAST(season AS INTEGER), CAST(round AS INTEGER)
    LIMIT 10
""").pl()  # .pl() returns a Polars DataFrame

## 4. Delta time travel

Every write to a Delta table creates a new version. You can read any past version — useful when debugging "why is this row different from yesterday?".

In [ ]:
dt = DeltaTable(str(RAW_RACES_PATH))

print(f"Current version: {dt.version()}")
print("\nHistory (oldest first):")
for entry in dt.history():
    print(f"  v{entry['version']}: {entry.get('operation', '?')} at {entry.get('timestamp', '?')}")

In [ ]:
# Read a specific past version (only meaningful after multiple materializations)
# pl.read_delta(str(RAW_RACES_PATH), version=0)

## 5. Silver layer — what staging produced

Three staging models cleaned and conformed the raw layer. Each is a single Parquet file.

> Run `Materialize` on the staging assets (or any downstream mart) in Dagster first — these cells will error if the files don't exist yet.

In [3]:
# 24 drivers from the 2024 grid — typed dates, snake_case names, convenience full_name column
stg_drivers = pl.read_parquet(STG_DRIVERS_PATH)
print(f"Rows: {stg_drivers.height}")
stg_drivers.head()

Rows: 25


driver_id,permanent_number,driver_code,given_name,family_name,full_name,date_of_birth,nationality,driver_url
str,i32,str,str,str,str,date,str,str
"""albon""",23,"""ALB""","""Alexander""","""Albon""","""Alexander Albon""",1996-03-23,"""Thai""","""http://en.wikipedia.org/wiki/A…"
"""alonso""",14,"""ALO""","""Fernando""","""Alonso""","""Fernando Alonso""",1981-07-29,"""Spanish""","""http://en.wikipedia.org/wiki/F…"
"""antonelli""",12,"""ANT""","""Andrea Kimi""","""Antonelli""","""Andrea Kimi Antonelli""",2006-08-25,"""Italian""","""https://en.wikipedia.org/wiki/…"
"""bearman""",87,"""BEA""","""Oliver""","""Bearman""","""Oliver Bearman""",2005-05-08,"""British""","""http://en.wikipedia.org/wiki/O…"
"""bottas""",77,"""BOT""","""Valtteri""","""Bottas""","""Valtteri Bottas""",1989-08-28,"""Finnish""","""http://en.wikipedia.org/wiki/V…"


In [4]:
# ~480 results: one row per (race, driver) finishing position
stg_results = pl.read_parquet(STG_RESULTS_PATH)
print(f"Rows: {stg_results.height}")
stg_results.head()

Rows: 6075


season,round,race_name,driver_id,constructor_id,constructor_name,car_number,position_raw,finishing_position,points,grid_position,laps_completed,finish_status,race_time_ms,fastest_lap_rank,fastest_lap_number
i32,i32,str,str,str,str,i32,str,i32,f64,i32,i32,str,i64,i32,i32
2024,1,"""Bahrain Grand Prix""","""max_verstappen""","""red_bull""","""Red Bull""",1,"""1""",1,26.0,1,57,"""Finished""",5504742,1,39
2024,1,"""Bahrain Grand Prix""","""perez""","""red_bull""","""Red Bull""",11,"""2""",2,18.0,5,57,"""Finished""",5527199,4,40
2024,1,"""Bahrain Grand Prix""","""sainz""","""ferrari""","""Ferrari""",55,"""3""",3,15.0,4,57,"""Finished""",5529852,6,44
2024,1,"""Bahrain Grand Prix""","""leclerc""","""ferrari""","""Ferrari""",16,"""4""",4,12.0,2,57,"""Finished""",5544411,2,36
2024,1,"""Bahrain Grand Prix""","""russell""","""mercedes""","""Mercedes""",63,"""5""",5,10.0,3,57,"""Finished""",5551530,12,40


## 6. Gold layer — the marts

Two marts answer business questions. They aggregate across staging models and change the grain (per-country, per-driver).

In [5]:
# mart_country_race_summary — one row per country, sorted by race count
country_summary = pl.read_parquet(MART_COUNTRIES_PATH).sort("race_count", descending=True)
country_summary

country,race_count,circuits,first_race_date,latest_race_date,centroid_lat,centroid_long
str,i64,str,date,date,f64,f64
"""USA""",3,"""Miami International Autodrome,…",2024-05-05,2024-11-23,30.7352,-97.684333
"""Italy""",2,"""Autodromo Nazionale di Monza, …",2024-05-19,2024-09-01,44.97975,10.498905
"""Australia""",1,"""Albert Park Grand Prix Circuit""",2024-03-24,2024-03-24,-37.8497,144.968
"""Austria""",1,"""Red Bull Ring""",2024-06-30,2024-06-30,47.2197,14.7647
"""Azerbaijan""",1,"""Baku City Circuit""",2024-09-15,2024-09-15,40.3725,49.8533
…,…,…,…,…,…,…
"""Saudi Arabia""",1,"""Jeddah Corniche Circuit""",2024-03-09,2024-03-09,21.6319,39.1044
"""Singapore""",1,"""Marina Bay Street Circuit""",2024-09-22,2024-09-22,1.2914,103.864
"""Spain""",1,"""Circuit de Barcelona-Catalunya""",2024-06-23,2024-06-23,41.57,2.26111


In [6]:
# mart_driver_standings — championship-style summary, sorted by points
standings = pl.read_parquet(MART_STANDINGS_PATH)
with pl.Config(tbl_cols=-1, tbl_width_chars=200):
    print(standings)

shape: (24, 14)
┌────────────────┬────────────────┬─────────────┬───────────────┬───────────────┬───────────────┬──────────────┬───────────────┬──────┬─────────┬───────────────┬──────┬───────────────┬───────────────┐
│ driver_id      ┆ driver_name    ┆ driver_code ┆ nationality   ┆ date_of_birth ┆ primary_const ┆ total_points ┆ races_entered ┆ wins ┆ podiums ┆ points_finish ┆ dnfs ┆ pole_position ┆ avg_finishing │
│ ---            ┆ ---            ┆ ---         ┆ ---           ┆ ---           ┆ ructor        ┆ ---          ┆ ---           ┆ ---  ┆ ---     ┆ es            ┆ ---  ┆ s             ┆ _position     │
│ str            ┆ str            ┆ str         ┆ str           ┆ date          ┆ ---           ┆ f64          ┆ i64           ┆ f64  ┆ f64     ┆ ---           ┆ f64  ┆ ---           ┆ ---           │
│                ┆                ┆             ┆               ┆               ┆ str           ┆              ┆               ┆      ┆         ┆ f64           ┆      ┆ f64        

## 7. Ad-hoc analysis — Polars + DuckDB sandbox

You don't have to go through dbt to explore the data. Polars joins across the staging Parquet files directly, and DuckDB can SQL them. Use this for quick experiments before deciding what becomes a permanent dbt model.

In [7]:
# Example: total championship points by driver nationality.
# Pure Polars — joins stg_results + stg_drivers without going through the marts.
points_by_nation = (
    stg_results
    .join(stg_drivers, on="driver_id")
    .group_by("nationality")
    .agg(pl.col("points").sum().alias("total_points"))
    .sort("total_points", descending=True)
)
points_by_nation.head(10)

nationality,total_points
str,f64
"""British""",9520.0
"""Dutch""",4469.0
"""Spanish""",3950.0
"""Monegasque""",3846.0
"""Australian""",3286.0
"""Mexican""",1479.0
"""French""",802.0
"""German""",482.0
"""Japanese""",378.0
